# 5 · In-memory PDF retrieval and generation

The included policy manual identifies the OpenGov Foundation and is dated January 8, 2018. It is historical reference material, not your employer's current policy. Install `requirements-rag.txt`. Configure the root `.env` before running Azure cells. Requests may incur charges. Stored outputs are cleared; no new live Azure results are supplied.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
from lab.config import get_chat_model

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from lab.retrieval import load_pdf_chunks, get_embeddings, SearchHit, answer_question
documents = load_pdf_chunks(ROOT / "assets/hr_policy_manual.pdf")
embeddings = get_embeddings()
store = InMemoryVectorStore(embeddings)
store.add_documents(documents)
print("Indexed chunks:", len(documents))

## Retrieve, inspect, then answer
A cosine cutoff is optional and is not calibrated confidence. Filtering is applied explicitly; passing a score threshold to a plain similarity retriever does not necessarily enable filtering.

In [ ]:
question = "What does the supplied manual say about alcohol?"
minimum_score = None  # Set a cosine cutoff only after inspecting scores on representative queries.
matches = store.similarity_search_with_score(question, k=4)
hits = [SearchHit(document, score) for document, score in matches
        if minimum_score is None or score >= minimum_score]
for hit in hits:
    print(hit.document.metadata, round(hit.score, 3))
print(answer_question(question, hits))

In [ ]:
question = "Where should I stay in New Delhi?"
matches = store.similarity_search_with_score(question, k=4)
hits = [SearchHit(document, score) for document, score in matches
        if minimum_score is None or score >= minimum_score]
print(answer_question(question, hits))

The second question is intentionally outside the manual's subject. Nearest-neighbor search can still return passages. Inspect whether the answer says it does not know; the prompt alone cannot guarantee abstention. No live model outputs or accuracy scores are supplied.